In [1]:
# imports
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, date
import warnings
warnings.filterwarnings('ignore')

In [17]:
customer_path = "/Users/farhan/Desktop/Courses/ace-bikes-data-transformation/data_original/Customers.xlsx"
order_path = "/Users/farhan/Desktop/Courses/ace-bikes-data-transformation/data_original/OrderInfo.xlsx"
employee_path = "/Users/farhan/Desktop/Courses/ace-bikes-data-transformation/data_original/Employees.xlsx"
inventory_path = "/Users/farhan/Desktop/Courses/ace-bikes-data-transformation/data_original/Inventory.xlsx"

In [18]:
customers = pd.read_excel(customer_path)
orders = pd.read_excel(order_path)
employees = pd.read_excel(employee_path)
inventory = pd.read_excel(inventory_path)

In [4]:
orders.head(3)

,CustomerID,LocationID,Date,Time,EmployeeID,OrderID
0,445,L01,2017-02-23,14:56:00,3,1
1,450,L01,2017-01-03,16:53:00,3,2
2,445,L01,2017-01-04,14:11:00,3,3


In [5]:
customers.head(3)

,id,first_name,last_name,gender,DOB,LoyaltyMember,EmailList,Source
0,1,Eveleen,Erat,F,1977-02-01,1.0,1.0,Newspaper
1,2,Micheil,Fransseni,M,1980-06-17,1.0,0.0,Social
2,4,Carin,Oulett,F,1973-03-14,1.0,0.0,Referral


In [6]:
employees.head(3)

,Employee Number,FName,LName,gender,SkillsTraining,SalesmanshipTraining,ProductTraining,DOB
0,1,Warden,Oylett,Male,False,False,False,1996-08-21
1,2,Tulley,Cockroft,Male,True,True,True,1997-08-04
2,3,Pip,Cuming,Male,False,True,False,1993-02-05


In [7]:
customer_orders = orders.merge(
    customers,
    left_on="CustomerID",
    right_on="id",
    how="inner"
).copy()

employee_orders = orders.merge(
    employees,
    left_on="EmployeeID",
    right_on="Employee Number",
    how="inner"
)

In [8]:
customer_orders["Date"] = pd.to_datetime(customer_orders["Date"])
employee_orders["Date"] = pd.to_datetime(employee_orders["Date"])

In [9]:
# Keep a copy of the 2022 data for later use
customer_orders_2022 = customer_orders[customer_orders["Date"].dt.year == 2022].copy()
employee_orders_2022 = employee_orders[employee_orders["Date"].dt.year == 2022].copy()

customer_orders = customer_orders[customer_orders["Date"].dt.year != 2022]
employee_orders = employee_orders[employee_orders["Date"].dt.year != 2022]

In [10]:
customers = customer_orders[["id", "first_name", "last_name", "gender", "DOB", "LoyaltyMember", "EmailList", "Source"]].drop_duplicates()
orders = customer_orders[["CustomerID", "LocationID", "Date", "Time", "EmployeeID", "OrderID"]]
employees = employee_orders[[
    "EmployeeID", "FName", "LName", "gender", 
    "SkillsTraining", "SalesmanshipTraining", "ProductTraining",
    "DOB"
]].drop_duplicates()

In [11]:
# save the 2022 data please 
customer_orders_2022.to_excel("../archive/customer_orders_2022.xlsx", index=False)
employee_orders_2022.to_excel("../archive/employee_orders_2022.xlsx", index=False)

In [12]:
# save as xlsx overriting the original files
customers.to_excel(customer_path, index=False)
orders.to_excel(order_path, index=False)
employees.to_excel(employee_path, index=False)

In [14]:
line_items = pd.read_csv("../data_original/LineItemSales.csv")
line_items.head(3)

,LineItemID,OrderID,ItemID,Qty,DiscountID
0,1,1,20,1,D3
1,2,1,2,1,NaN
2,3,1,53,1,NaN


In [15]:
orders_id_list_2022 = customer_orders_2022["OrderID"].unique()
line_items_2022 = line_items[line_items["OrderID"].isin(orders_id_list_2022)].copy()
line_items_2022.to_csv("../archive/line_items_2022.csv", index=False)

# Get line items other than 2022
line_items = line_items[~line_items["OrderID"].isin(orders_id_list_2022)].copy()
line_items.to_csv("../data_original/LineItemSales.csv", index=False)

In [19]:
inventory.head(3)

,InventoryCountID,LocationID,ItemID,2017-01-01 00:00:00,2017-02-01 00:00:00,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,...,2022-03-01 00:00:00,2022-04-01 00:00:00,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00
0,1,L01,1,4.0,3.0,2.0,0.0,3.0,1.0,0.0,...,1,2,1,1,1,2,2,5,4,5
1,2,L01,2,4.0,5.0,0.0,1.0,5.0,1.0,3.0,...,0,2,5,0,1,3,1,0,2,2
2,3,L01,3,2.0,4.0,4.0,4.0,0.0,0.0,5.0,...,4,0,4,1,5,5,3,5,1,3


In [20]:
cols_to_drop = [col for col in inventory.columns if str(col).startswith("2022-")]
inventory = inventory.drop(columns=cols_to_drop)

In [21]:
# save as xlsx overriting the original files
inventory.to_excel(inventory_path, index=False)